# True-LoRA: matutake にコーディング能力を付与するチュートリアル

**Target Model:** [summerMC/matutake](https://huggingface.co/summerMC/matutake) (2B params, Qwen2)

**Goal:** True-LoRAの検索型アダプタ生成を使って、matutakeにPythonコーディング能力を付与するLoRAアダプタを作成し、ベースモデルにマージします。

**Hardware:** Colab Free Tier (T4 GPU)

---

### 流れ

1. 環境セットアップ
2. モデルの準備（アーキテクチャ解析）
3. コーディングアダプタバンクの構築
4. True-LoRA トレーニング
5. LoRAアダプタの生成
6. マージ（ベースモデルに統合）
7. テスト
8. HuggingFace Hubにアップロード（オプション）

---
## Step 1: 環境セットアップ

In [ ]:
# GPU確認
!nvidia-smi

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name}")
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")

In [ ]:
# True-LoRAをインストール
!pip install git+https://github.com/MARVserver/TrueLora.git -q
!pip install transformers accelerate -q

print("Done!")

In [ ]:
import os, json, math
from pathlib import Path
from dataclasses import dataclass

import torch
import torch.nn.functional as F
from torch import nn

from true_lora.adapter import (
    AdapterBank, AdapterSpec, LoraTensorSpec,
    save_peft_adapter, adapter_fingerprint,
)
from true_lora.generator import TrueLoraGenerator
from true_lora.text import HashingTextEncoder
from true_lora.train import train_on_adapter_bank
from true_lora.merge import merge_adapter_into_hf_model
from true_lora.reporting import write_json_report

from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

---
## Step 2: モデルの準備

In [ ]:
# モデル設定
MODEL_NAME = "summerMC/matutake"
WORK_DIR = Path("./truelora_work")
WORK_DIR.mkdir(exist_ok=True)
OUTPUT_DIR = WORK_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

# LoRAパラメータ
LORA_RANK = 8
LORA_ALPHA = 16.0
TEXT_DIM = 256
HIDDEN_DIM = 512
MAX_NORM = 4.0

# Qwen2のLoRAターゲットモジュール
LORA_TARGETS = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

# モデル設定を動的に取得
config = AutoConfig.from_pretrained(MODEL_NAME, trust_remote_code=True)
HIDDEN_SIZE = config.hidden_size
NUM_LAYERS = config.num_hidden_layers

# 対象レイヤー（全層のサブセット）
TARGET_LAYERS = list(range(0, NUM_LAYERS, 4))

print(f"Model: {MODEL_NAME}")
print(f"Hidden size: {HIDDEN_SIZE}")
print(f"Num layers: {NUM_LAYERS}")
print(f"Target layers: {TARGET_LAYERS}")
print(f"LoRA targets/layer: {LORA_TARGETS}")

In [ ]:
def create_qwen2_lora_specs(
    hidden_size: int,
    target_layers: list[int],
    lora_targets: list[str],
    rank: int = 8,
    alpha: float = 16.0,
) -> list[LoraTensorSpec]:
    specs = []
    for layer_idx in target_layers:
        for target in lora_targets:
            if target in ["q_proj", "k_proj", "v_proj", "o_proj"]:
                name = f"model.layers.{layer_idx}.self_attn.{target}"
                out_feat = hidden_size
                in_feat = hidden_size
            elif target in ["gate_proj", "up_proj"]:
                name = f"model.layers.{layer_idx}.mlp.{target}"
                out_feat = hidden_size * 4
                in_feat = hidden_size
            elif target == "down_proj":
                name = f"model.layers.{layer_idx}.mlp.{target}"
                out_feat = hidden_size
                in_feat = hidden_size * 4
            else:
                continue
            specs.append(LoraTensorSpec(name, out_features=out_feat, in_features=in_feat, rank=rank, alpha=alpha))
    return specs

lora_specs = create_qwen2_lora_specs(HIDDEN_SIZE, TARGET_LAYERS, LORA_TARGETS, LORA_RANK, LORA_ALPHA)

print(f"LoRA specs: {len(lora_specs)}")
for spec in lora_specs[:3]:
    print(f"  {spec.name}: A={spec.a_shape}, B={spec.b_shape}")
print("  ...")

---
## Step 3: コーディングアダプタバンクの構築

True-LoRAは、事前定義されたアダプタバンクから最適なアダプタを検索してLoRAを生成します。
ここではコーディングタスク向けのアダプタバンクを作成します。

In [ ]:
encoder = HashingTextEncoder(dim=TEXT_DIM)

coding_adapters_config = [
    {"desc": "python code generation", "sa": 0.15, "sb": 0.08, "score": 0.85},
    {"desc": "python function implementation", "sa": 0.18, "sb": 0.09, "score": 0.82},
    {"desc": "python class and module design", "sa": 0.12, "sb": 0.06, "score": 0.78},
    {"desc": "python list comprehension and generators", "sa": 0.14, "sb": 0.07, "score": 0.75},
    {"desc": "python error handling and exceptions", "sa": 0.16, "sb": 0.08, "score": 0.80},
    {"desc": "algorithm implementation in python", "sa": 0.20, "sb": 0.10, "score": 0.88},
    {"desc": "data structure implementation", "sa": 0.17, "sb": 0.08, "score": 0.83},
    {"desc": "dynamic programming solutions", "sa": 0.22, "sb": 0.11, "score": 0.86},
    {"desc": "recursive algorithm design", "sa": 0.19, "sb": 0.09, "score": 0.81},
    {"desc": "graph algorithm implementation", "sa": 0.21, "sb": 0.10, "score": 0.84},
    {"desc": "code debugging and error fixing", "sa": 0.13, "sb": 0.06, "score": 0.79},
    {"desc": "unit test writing with pytest", "sa": 0.15, "sb": 0.07, "score": 0.77},
    {"desc": "code review and improvement", "sa": 0.14, "sb": 0.07, "score": 0.76},
    {"desc": "performance optimization", "sa": 0.18, "sb": 0.09, "score": 0.80},
    {"desc": "code refactoring and cleanup", "sa": 0.12, "sb": 0.06, "score": 0.74},
    {"desc": "numpy array operations", "sa": 0.16, "sb": 0.08, "score": 0.82},
    {"desc": "pandas data manipulation", "sa": 0.14, "sb": 0.07, "score": 0.79},
    {"desc": "torch deep learning code", "sa": 0.19, "sb": 0.09, "score": 0.85},
    {"desc": "requests and api integration", "sa": 0.11, "sb": 0.05, "score": 0.73},
    {"desc": "file io and pathlib usage", "sa": 0.10, "sb": 0.05, "score": 0.71},
    {"desc": "design pattern implementation", "sa": 0.17, "sb": 0.08, "score": 0.78},
    {"desc": "object oriented programming", "sa": 0.15, "sb": 0.07, "score": 0.76},
    {"desc": "functional programming style", "sa": 0.13, "sb": 0.06, "score": 0.74},
    {"desc": "context manager and decorator", "sa": 0.14, "sb": 0.07, "score": 0.77},
    {"desc": "type hints and annotations", "sa": 0.11, "sb": 0.05, "score": 0.72},
    {"desc": "secure coding practices", "sa": 0.12, "sb": 0.06, "score": 0.75},
    {"desc": "input validation and sanitization", "sa": 0.13, "sb": 0.06, "score": 0.76},
    {"desc": "authentication and authorization code", "sa": 0.14, "sb": 0.07, "score": 0.78},
]

print(f"Defined {len(coding_adapters_config)} coding adapters")

In [ ]:
# アダプタバンクを作成
encoder = HashingTextEncoder(dim=TEXT_DIM)
adapters = []

for cfg in coding_adapters_config:
    tensors = {}
    for spec in lora_specs:
        tensors[f"{spec.name}.lora_A.weight"] = torch.full(spec.a_shape, cfg["sa"])
        tensors[f"{spec.name}.lora_B.weight"] = torch.full(spec.b_shape, cfg["sb"])

    adapter = AdapterSpec(
        description=cfg["desc"],
        embedding=encoder.encode(cfg["desc"]),
        tensors=tensors,
        metrics={"score": cfg["score"]},
    )
    adapters.append(adapter)

bank = AdapterBank(adapters)

print(f"Adapter bank created: {len(bank.adapters)} adapters")
print(f"Sample fingerprints:")
for i, a in enumerate(bank.adapters[:3]):
    print(f"  [{i}] {a.description} score={a.metrics['score']:.2f}")
print("  ...")

---
## Step 4: True-LoRA トレーニング

検索型LoRA生成器を学習します。
- **入力:** テキストプロンプト
- **出力:** 最適なアダプタの重みを組み合わせたLoRAテンサー

In [ ]:
# True-LoRA生成器を作成
generator = TrueLoraGenerator(
    tensor_specs=lora_specs,
    adapter_bank=bank,
    text_dim=TEXT_DIM,
    hidden_dim=HIDDEN_DIM,
    max_tensor_norm=MAX_NORM,
)

total_params = sum(p.numel() for p in generator.hyper.parameters())
print(f"Generator created")
print(f"  HyperAdapter params: {total_params:,}")
print(f"  LoRA specs: {len(lora_specs)}")
print(f"  Adapter bank: {len(bank.adapters)} adapters")

In [ ]:
# トレーニング
print("Training started...")
print("(200 steps on Colab Free Tier, ~2-3 min)\n")

losses = train_on_adapter_bank(
    generator,
    bank.adapters,
    steps=200,
    lr=1e-3,
)

print(f"\nTraining complete!")
print(f"  Final loss: {losses[-1]:.6f}")
print(f"  Loss range: [{min(losses):.6f}, {max(losses):.6f}]")

In [ ]:
# トレーニング済みチェックポイントを保存
checkpoint_path = WORK_DIR / "generator_checkpoint.pt"
torch.save({
    "hyper_state_dict": generator.hyper.state_dict(),
    "tensor_specs": [{"name": s.name, "out_features": s.out_features, "in_features": s.in_features, "rank": s.rank, "alpha": s.alpha} for s in lora_specs],
    "text_dim": TEXT_DIM,
    "hidden_dim": HIDDEN_DIM,
    "max_tensor_norm": MAX_NORM,
}, checkpoint_path)
print(f"Checkpoint saved: {checkpoint_path}")

---
## Step 5: LoRAアダプタの生成

「Python code generation」用のLoRAアダプタを生成します。

In [ ]:
# LoRAを生成
prompt = "python code generation"
print(f"Generating LoRA for: '{prompt}'\n")

state_dict, report = generator.generate(
    prompt,
    retrieval_k=8,
)

print(f"Uncertainty: {report['uncertainty']:.4f}")
print(f"Abstained: {report['abstained']:.4f}")
print(f"Max retrieval score: {report['max_retrieval_score']:.4f}")
print(f"\nRetrieved adapters:")
for provenance in report["retrieved_adapters"]:
    print(f"  [{provenance['rank']}] {provenance['description']} (w={provenance['weight']:.3f})")
print(f"\nTensor count: {len(state_dict)}")

In [ ]:
# LoRAアダプタを保存
adapter_path = OUTPUT_DIR / "coding_lora.pt"
save_peft_adapter(adapter_path, state_dict, report)
print(f"LoRA adapter saved: {adapter_path}")

# レポートも保存
report_path = OUTPUT_DIR / "generation_report.json"
write_json_report(report_path, report)
print(f"Report saved: {report_path}")

---
## Step 6: マージ

生成したLoRAアダプタをmatutakeベースモデルにマージします。

In [ ]:
merged_dir = OUTPUT_DIR / "matutake_coding_merged"
print(f"Merging LoRA into {MODEL_NAME}...")
print("(Downloading model + merging, ~5-10 min)\n")

merge_report = merge_adapter_into_hf_model(
    adapter_path=adapter_path,
    model_name_or_path=MODEL_NAME,
    output_dir=merged_dir,
    dtype="bfloat16",
    device="cpu",
    allow_download=True,
)

print(f"\nMerge complete!")
print(f"Output: {merge_report['output_dir']}")
print(f"Applied layers: {merge_report['applied_layers']}")

---
## Step 7: テスト

マージしたモデルのコーディング能力を検証します。

In [ ]:
# マージしたモデルを読み込み
print("Loading merged model...")

tokenizer = AutoTokenizer.from_pretrained(str(merged_dir), trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    str(merged_dir),
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

params = sum(p.numel() for p in model.parameters())
print(f"Model loaded! Parameters: {params / 1e9:.2f}B")

In [ ]:
def test_coding(model, tokenizer, user_prompt, max_new_tokens=512):
    messages = [
        {"role": "system", "content": "You are a helpful coding assistant. Write clean, efficient Python code."},
        {"role": "user", "content": user_prompt},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.05,
        )
    return tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

In [ ]:
# Test 1: 関数実装
print("=" * 60)
print("Test 1: Binary Search")
print("=" * 60)
print(test_coding(model, tokenizer,
    "Write a Python function that implements binary search on a sorted array. Include type hints and docstring."))

In [ ]:
# Test 2: アルゴリズム
print("=" * 60)
print("Test 2: Dynamic Programming")
print("=" * 60)
print(test_coding(model, tokenizer,
    "Implement a function to find the longest common subsequence of two strings using dynamic programming."))

In [ ]:
# Test 3: 日本語
print("=" * 60)
print("Test 3: Japanese coding")
print("=" * 60)
print(test_coding(model, tokenizer,
    "Pythonで、リストの中から重複を除いた要素を返す関数を書いてください。メソッドを使わずに実装してください。"))

---
## Step 8: HuggingFace Hubにアップロード（オプション）

アップロードするには HuggingFace のトークンが必要です。

In [ ]:
USE_HF_UPLOAD = False  # Trueに変更するとアップロード
HF_REPO = "summerMC/matutake-coding-lora"

if USE_HF_UPLOAD:
    from huggingface_hub import login, HfApi
    login()  # トークンを入力
    api = HfApi()
    api.upload_folder(folder_path=str(merged_dir), repo_id=HF_REPO, repo_type="model")
    print(f"Uploaded: https://huggingface.co/{HF_REPO}")
else:
    print("Upload skipped. Set USE_HF_UPLOAD = True to upload.")
    print(f"Manual upload: huggingface-cli upload {HF_REPO} {merged_dir} .")

---
## まとめ

### 作成ファイル

| ファイル | 説明 |
| --- | --- |
| `truelora_work/output/coding_lora.pt` | 生成されたLoRAアダプタ |
| `truelora_work/output/matutake_coding_merged/` | マージ済みモデル |
| `truelora_work/generator_checkpoint.pt` | トレーニング済み生成器 |

### パラメータ

| 項目 | 値 |
| --- | --- |
| LoRA Rank | 8 |
| LoRA Alpha | 16.0 |
| Target Layers | 7 (0, 4, 8, 12, 16, 20, 24) |
| LoRA Targets | 7/layer (q, k, v, o, gate, up, down) |
| Total LoRA Specs | 49 |
| Adapter Bank | 29 adapters |
| Training Steps | 200 |

### 使用例

```python
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model = AutoModelForCausalLM.from_pretrained(
    "summerMC/matutake-coding-lora",
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained("summerMC/matutake-coding-lora")

messages = [
    {"role": "system", "content": "You are a helpful coding assistant."},
    {"role": "user", "content": "Write a Python function to sort a list."},
]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors="pt").to(model.device)
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=512)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True))
```